# Bias in Machine Learning

Kita akan menjalankan sebuah eksperimen: melatih model rekrutmen sederhana pada dataset **synthetic** (buatan) yang sengaja mengandung bias historis, lalu melihat perilakunya.

Notebook ini lebih ke observasi dan diskusi — jalankan cell-nya, lalu diskusikan bersama.

> **Dataset ini 100% synthetic.** Tidak merepresentasikan individu atau perusahaan nyata.

## 1. Load dataset

In [1]:
import pandas as pd

df = pd.read_excel("datasets/recruitment_biased.xlsx")

print(f"Jumlah kandidat: {len(df)}")
print(f"Distribusi hired: {df['hired'].value_counts().to_dict()}")
print(f"Acceptance rate: {df['hired'].mean()*100:.2f}%")
df.head()

Jumlah kandidat: 1000
Distribusi hired: {0: 910, 1: 90}
Acceptance rate: 9.00%


,candidate_id,name,gender,years_experience,education_level,num_skills,previous_companies,portfolio_score,communication_score,hired
0,CAND-0001,Bella,F,6,S1,5,2,5.9,6.3,0
1,CAND-0002,Gunawan,M,2,S1,6,2,5.9,3.0,0
2,CAND-0003,Anita,F,6,D3,3,1,7.9,1.9,0
3,CAND-0004,Adit,M,7,S1,6,2,9.8,4.4,0
4,CAND-0005,Rina,F,4,S2,2,0,3.1,3.6,0


**Diskusi 1:**  
Class distribution di sini sekitar 10% hired, 90% rejected. Ingat pelajaran 4.3 — apakah accuracy saja cukup untuk mengevaluasi model di dataset seperti ini? Kenapa?

## 2. Distribusi hiring per gender

In [2]:
gender_stats = df.groupby('gender').agg(
    total_kandidat=('hired', 'size'),
    total_hired=('hired', 'sum'),
    hire_rate=('hired', 'mean'),
)
gender_stats['hire_rate'] = (gender_stats['hire_rate'] * 100).round(2)
print(gender_stats)

        total_kandidat  total_hired  hire_rate
gender                                        
F                  448           10       2.23
M                  552           80      14.49


**Diskusi 2:**  
Sudah ada gap besar di data mentah. **Bias-nya bukan dari model. Bias-nya dari data.**  
Kalau kita training model pada data ini apa adanya, apa yang akan model pelajari?

## 3. Training model
Kita gunakan Logistic Regression sederhana. Perhatikan bahwa fitur `gender` juga masuk sebagai input.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

feature_cols = [
    'gender', 'years_experience', 'education_level',
    'num_skills', 'previous_companies',
    'portfolio_score', 'communication_score'
]

df_encoded = pd.get_dummies(df[feature_cols], columns=['gender', 'education_level'], drop_first=False)
X = df_encoded
y = df['hired']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_s, y_train)

print("Model trained.")

Model trained.


## 4. Evaluasi — metrik overall

In [4]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test_s)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, digits=4))

Accuracy: 0.8150
              precision    recall  f1-score   support

           0     0.9866    0.8077    0.8882       182
           1     0.3137    0.8889    0.4638        18

    accuracy                         0.8150       200
   macro avg     0.6502    0.8483    0.6760       200
weighted avg     0.9260    0.8150    0.8500       200



**Diskusi 3:**  
Angka-angkanya kelihatan lumayan. Kalau kamu hanya melihat metrik ini di dashboard, apakah kamu akan approve model ini deploy ke production?

## 5. Test kritis — dua resume identik, hanya gender berbeda

In [5]:
def make_candidate(gender, years_experience=5, education_level='S1',
                   num_skills=6, previous_companies=2,
                   portfolio_score=8.0, communication_score=8.0):
    row = {
        'gender': gender,
        'years_experience': years_experience,
        'education_level': education_level,
        'num_skills': num_skills,
        'previous_companies': previous_companies,
        'portfolio_score': portfolio_score,
        'communication_score': communication_score,
    }
    row_df = pd.DataFrame([row])
    encoded = pd.get_dummies(row_df, columns=['gender', 'education_level'], drop_first=False)
    for col in X.columns:
        if col not in encoded.columns:
            encoded[col] = 0
    encoded = encoded[X.columns]
    return scaler.transform(encoded)


andi  = make_candidate(gender='M')
anita = make_candidate(gender='F')

print("=== Andi (M) — 5th exp, S1, 6 skills, 2 companies, portfolio 8, comm 8 ===")
print(f"  Hired probability: {model.predict_proba(andi)[0][1]:.4f}")
print(f"  Predicted        : {'HIRED' if model.predict(andi)[0] == 1 else 'REJECTED'}")
print()
print("=== Anita (F) — 5th exp, S1, 6 skills, 2 companies, portfolio 8, comm 8 ===")
print(f"  Hired probability: {model.predict_proba(anita)[0][1]:.4f}")
print(f"  Predicted        : {'HIRED' if model.predict(anita)[0] == 1 else 'REJECTED'}")

=== Andi (M) — 5th exp, S1, 6 skills, 2 companies, portfolio 8, comm 8 ===
  Hired probability: 0.9197
  Predicted        : HIRED

=== Anita (F) — 5th exp, S1, 6 skills, 2 companies, portfolio 8, comm 8 ===
  Hired probability: 0.2647
  Predicted        : REJECTED


**Diskusi 4 (kunci):**  
Dua kandidat, kualifikasi identik, hanya sinyal gender berbeda. Prediksi berbeda.

- Apa yang barusan model lakukan?  
- Apakah ini adil?  
- Apakah model "tahu" bahwa ini tidak adil? Kenapa?

## 6. Cek performance per gender

In [6]:
from sklearn.metrics import precision_score, recall_score

test_idx = X_test.index
test_gender = df.loc[test_idx, 'gender']

print(f"{'Gender':<10} {'Support':<10} {'Accuracy':<12} {'Precision':<12} {'Recall':<12}")
print("-" * 60)

for g in ['M', 'F']:
    mask = test_gender == g
    y_true_g = y_test[mask]
    y_pred_g = y_pred[mask.values]

    if len(y_true_g) == 0:
        continue

    acc_g = accuracy_score(y_true_g, y_pred_g)
    prec_g = precision_score(y_true_g, y_pred_g, zero_division=0)
    rec_g = recall_score(y_true_g, y_pred_g, zero_division=0)

    print(f"{g:<10} {len(y_true_g):<10} {acc_g:<12.4f} {prec_g:<12.4f} {rec_g:<12.4f}")

Gender     Support    Accuracy     Precision    Recall      
------------------------------------------------------------
M          122        0.7377       0.3023       0.8667      
F          78         0.9359       0.3750       1.0000      
